### Notebook 范围

### 目的
扫描 hindcast case 和 cleaned 产品是否存在，并建立总览矩阵。

### 科学问题
哪些年份、初始化月份、INT/NOCOUPL、扰动类型可用于 source diagnosis 或反馈对比？

### 方法
扫描 /mnt/soclim0/public_data/weiji/Hindcast，识别 partial_O3、EPflux_daily_ubar、U、Z3、FWD、AO/NAM。

### 输出
outputs/figures/00_overview 和 outputs/tables/00_overview。

### 解读
绿色/蓝色表示可用配置，0003 标记为特殊分支。

### 注意
这是产品级 inventory；单成员缺失会在后续 notebook 的 logs 中记录。


### 导入与公共路径

### 目的
加载 Extention_analysis 的公共函数、数据路径、输出目录和绘图参数。

### 科学问题
保证所有 notebook 使用同一套变量定义，避免 EPFlux、O3、U、Z300 的窗口和符号约定互相漂移。

### 方法
使用 hindcast_extension_utils.py 中的 DATA_ROOT、HINDCAST_ROOT、BWCN_ROOT、B2000_ROOT、FIG_DIR、TAB_DIR、CACHE_DIR 和 LOG_DIR。

### 输出
无图；只打印工作目录和 Hindcast 数据目录。

### 解读
如果路径打印正确，后续代码块会使用同一套 cleaned hindcast 产品。

### 注意
所有写入都限制在 code_cleaned/Hindcast_experiment/Extention_analysis/outputs 下；不会修改原始数据。


In [ ]:
from pathlib import Path
import sys

WORK_ROOT = Path("/home/weiji/restart_exam/code_cleaned/Hindcast_experiment/Extention_analysis")
if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))

import math
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

import hindcast_extension_utils as hx
from hindcast_extension_utils import *

for d in [FIG_DIR, TAB_DIR, CACHE_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("WORK_ROOT", WORK_ROOT)
print("HINDCAST_ROOT", HINDCAST_ROOT)


### 绘制 availability matrix

### 目的
生成 case inventory 表和可用性矩阵。

### 科学问题
哪些 case 适合做多案例机制稳健性检验？哪些 case 适合 INT vs NOCOUPL？

### 方法
按 year-init_month 聚合 case，单元格写出 INT/NOCOUPL/v2/v3 和成员数。

### 输出
00_hindcast_availability_matrix.png/pdf 和 00_case_inventory.csv，放在 00_overview 子目录。

### 解读
同一年同月同时有 INT 与 NOCOUPL 时，可以用于反馈对比；只有 INT 时只能用于机制或路径检验。

### 注意
0003 是特殊分支，不强行配 NOCOUPL。

图中标签统一使用 `INT 3D`、`CLIM 3D`、`LP` 和 `MP`，其中 `LP` 对应 v2 large-temperature perturbation，`MP` 对应 v3 humidity perturbation。


In [ ]:
fig_dir = figure_dir("00_overview")
tab_dir = table_dir("00_overview")
inv = discover_hindcast_cases()
inv_path = tab_dir / "00_case_inventory.csv"
inv.to_csv(inv_path, index=False)
# Keep a root-level copy for downstream synthesis compatibility.
inv.to_csv(TAB_DIR / "00_case_inventory.csv", index=False)
print(inv[["case", "year", "init_month", "config", "perturbation", "n_members", "can_source_diagnose"]].to_string(index=False))

def _availability_label(row):
    """Return compact plot label for one hindcast experiment row."""
    config = row["config"]
    perturbation = row["perturbation"]
    n = int(row["n_members"])
    if config == "NOCOUPL":
        base = "CLIM 3D"
    else:
        base = "INT 3D"
        if perturbation == "v2_large_temperature":
            base += " LP"
        elif perturbation == "v3_humidity":
            base += " MP"
    return f"{base} n={n}"


def _availability_cell(sub, year):
    """Return face color and centered text for one year-month availability cell."""
    if sub.empty:
        return "#eeeeee", "missing"
    labels = []
    sub = sub.copy()
    order = {"small_temperature": 0, "v2_large_temperature": 1, "v3_humidity": 2}
    sub["_order"] = sub["perturbation"].map(order).fillna(9)
    sub["_config_order"] = sub["config"].map({"INT": 0, "NOCOUPL": 1}).fillna(9)
    for _, row in sub.sort_values(["_config_order", "_order", "case"]).iterrows():
        labels.append(_availability_label(row))
    if year == "0003":
        color = "#f4d7d4"
        labels.append("special branch")
    elif set(sub["config"]) >= {"INT", "NOCOUPL"}:
        color = "#c7e8b8"
    else:
        color = "#d8e8f8"
    return color, "\n".join(labels)


if inv.empty:
    fig = empty_figure("No hindcast cases found", "Hindcast availability")
else:
    years = SELECTED_YEARS
    months = ["01", "02", "03", "04"]
    month_labels = ["Jan", "Feb", "Mar", "Apr"]
    fig, ax = plt.subplots(figsize=(22, 10), constrained_layout=False)
    fig.subplots_adjust(left=0.09, right=0.985, top=0.90, bottom=0.13)
    ax.set_xlim(-0.5, len(months) - 0.5)
    ax.set_ylim(len(years) - 0.5, -0.5)
    ax.set_xticks(range(len(months)), month_labels)
    ax.set_yticks(range(len(years)), years)
    ax.set_xlabel("Initialization month", fontsize=18, labelpad=12)
    ax.set_ylabel("Reference year", fontsize=18, labelpad=12)
    ax.set_title("Hindcast availability matrix", fontsize=22, pad=16)
    ax.tick_params(axis="both", labelsize=18, width=1.4, length=7)

    for yi, year in enumerate(years):
        for mi, month in enumerate(months):
            sub = inv[(inv["year"] == year) & (inv["init_month"] == month)]
            color, text = _availability_cell(sub, year)
            ax.add_patch(
                plt.Rectangle(
                    (mi - 0.5, yi - 0.5),
                    1,
                    1,
                    facecolor=color,
                    edgecolor="#4f4f4f",
                    lw=1.35,
                    zorder=1,
                )
            )
            ax.text(mi, yi, text, ha="center", va="center", fontsize=13.5, linespacing=1.18, zorder=5)

    ax.grid(True, which="major", linestyle="--", color="0.78", linewidth=1.2, alpha=0.55, zorder=3)
    for spine in ax.spines.values():
        spine.set_linewidth(1.5)
        spine.set_color("black")

savefig(fig, "00_hindcast_availability_matrix", fig_dir=fig_dir, notebook="00_case_inventory_and_utils.ipynb", scientific_question="哪些 hindcast cases 可用于 source diagnosis 和 INT/NOCOUPL 对比？", variables_windows="case inventory; partial_O3, EPflux_daily_ubar, U, Z3, FWD, AO/NAM", interpretation="同一年同月 INT+NOCOUPL 表示可做 feedback comparison；0003 为特殊分支。", caveat="这是产品级检查，不代表每个变量每个成员都完整。", csv_table=inv_path)
plt.show()
write_figure_guide()
